In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.decomposition import PCA

In [75]:
def best_split(X, y):
    n_features = X.shape[1]
    best_rss = float('inf')
    best_feature = None
    best_threshold = None
    best_X_left = None
    best_y_left = None
    best_X_right = None
    best_y_right = None

    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])
        for threshold in thresholds:
            left_mask = X[:, feature_index] < threshold
            right_mask = ~left_mask
            X_left, y_left, X_right, y_right = X[left_mask], y[left_mask], X[right_mask], y[right_mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            rss = np.sum((y_left - np.mean(y_left))**2) + np.sum((y_right - np.mean(y_right))**2)

            if rss < best_rss:
                best_rss = rss
                best_feature = feature_index
                best_threshold = threshold
                best_X_left = X_left
                best_y_left = y_left
                best_X_right = X_right
                best_y_right = y_right

    return best_feature, best_threshold, best_X_left, best_y_left, best_X_right, best_y_right

def build_tree(X, y, max_depth=3, min_samples_split=2, depth=0):
    if len(y) < min_samples_split or depth >= max_depth or np.all(y == y[0]):
        return np.mean(y)

    best_feature, best_threshold, X_left, y_left, X_right, y_right = best_split(X, y)

    return {
        'feature': best_feature,
        'threshold': best_threshold,
        'left': build_tree(X_left, y_left, max_depth, min_samples_split, depth + 1),
        'right': build_tree(X_right, y_right, max_depth, min_samples_split, depth + 1)
    }

def predict_one(sample, tree):
    if not isinstance(tree, dict):
        return tree

    feature = tree['feature']
    threshold = tree['threshold']

    if sample[feature] < threshold:
        return predict_one(sample, tree['left'])
    else:
        return predict_one(sample, tree['right'])

def predict(X, tree):
    return np.array([predict_one(sample, tree) for sample in X])

def tune_tree(X, y, max_depth_list, min_samples_split_list, k=3):
    np.random.seed(1)

    best_mse = float('inf')
    best_params = None

    for max_depth in max_depth_list:
        for min_split in min_samples_split_list:
            indices = np.random.permutation(len(X))
            folds = np.array_split(indices, k)
            mse_list = []

            for i in range(k):
                val_idx = folds[i]
                train_idx = np.hstack([folds[j] for j in range(k) if j != i])

                X_train, y_train = X[train_idx], y[train_idx]
                X_val, y_val = X[val_idx], y[val_idx]
                
                tree = build_tree(X_train, y_train, max_depth=max_depth, min_samples_split=min_split)
                preds = predict(X_val, tree)
                mse = mean_squared_error(y_val, preds)
                mse_list.append(mse)

            avg_mse = np.mean(mse_list)
            print(f"depth={max_depth}, split={min_split} -> avg MSE: {avg_mse:.4f}")

            if avg_mse < best_mse:
                best_mse = avg_mse
                best_params = (max_depth, min_split)

    print("\nBest parameters:", best_params)
    print("Best CV average MSE:", best_mse)
    return best_params, best_mse

In [76]:
def clean_data(file):
    df = pd.read_csv(file)

    Y = df['ClaimNb']

    df['VehIsregular'] = (df['VehGas'] == 'Regular').astype(int)

    df = df.drop(columns= ['VehGas', 'IDpol', 'ClaimNb'])

    def letter_to_index(letter):
        return ord(letter.upper()) - ord('A')

    df['Area'] = df['Area'].apply(letter_to_index)

    df['Exposure'] = df['Exposure'].apply(lambda x: x - 1 if x > 1 else x)

    df = pd.get_dummies(df)

    scaler = StandardScaler()
    X = scaler.fit_transform(df)
   
    return X, Y

def split_data(X, Y, test_size):
    return train_test_split(X, Y, test_size=test_size, random_state=1)


In [91]:
X, Y = clean_data("claims_train.csv")

X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)

In [92]:
y_train = y_train.to_numpy()

In [96]:
subset_size = 50000  

subset_indices = np.random.choice(len(X_train), size=subset_size, replace=False)
X_train_subset = X_train[subset_indices]
y_train_subset = y_train[subset_indices]

In [98]:
best_params, best_mse = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[5, 7],
    min_samples_split_list=[1, 3, 5],
    k=3
)

depth=5, split=1 -> avg MSE: 0.0570
depth=5, split=3 -> avg MSE: 0.0571
depth=5, split=5 -> avg MSE: 0.0571
depth=7, split=1 -> avg MSE: 0.0586
depth=7, split=3 -> avg MSE: 0.0595
depth=7, split=5 -> avg MSE: 0.0593

Best parameters: (5, 1)
Best CV average MSE: 0.056989869953770624


In [ ]:
# checking 
best_params_1, best_mse_1 = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[2,3],
    min_samples_split_list=[2,3],
    k=2
)

tree_1 = build_tree(
    X_train_subset, y_train_subset,
    max_depth=best_params_1[0],
    min_samples_split=best_params_1[1]
)

best_params_2, best_mse_2 = tune_tree(
    X_train_subset, y_train_subset,
    max_depth_list=[2,3],
    min_samples_split_list=[2,3],
    k=2
)

tree_2 = build_tree(
    X_train_subset, y_train_subset,
    max_depth=best_params_2[0],
    min_samples_split=best_params_2[1]
)

import json
json.dumps(tree_2, sort_keys=True) == json.dumps(tree_1, sort_keys=True)


depth=2, split=2 -> avg MSE: 0.0400
depth=2, split=3 -> avg MSE: 0.0600
depth=3, split=2 -> avg MSE: 0.0200
depth=3, split=3 -> avg MSE: 0.0400

Best parameters: (3, 2)
Best CV average MSE: 0.02
depth=2, split=2 -> avg MSE: 0.0400
depth=2, split=3 -> avg MSE: 0.0600
depth=3, split=2 -> avg MSE: 0.0200
depth=3, split=3 -> avg MSE: 0.0400

Best parameters: (3, 2)
Best CV average MSE: 0.02


True

In [82]:
# best_params, best_mse = tune_tree(
#     X_train, y_train,
#     max_depth_list=[1, 2, 3, 4],
#     min_samples_split_list=[1, 2, 3, 4, 5],
#     # min_samples_split_list=[20, 50, 100, 1000],
#     k=3
# )

In [83]:
X = np.array([[1], [2], [10], [12]])
y = np.array([1, 1, 5, 5])
tree = build_tree(X, y, max_depth=3, min_samples_split=1)
print(tree)
preds = predict(X, tree)
print(preds)
threshold = 10
left = y[X[:,0] < threshold]
right = y[X[:,0] >= threshold]
print(left, right)
print(np.sum((left - left.mean())**2) + np.sum((right - right.mean())**2))


{'feature': 0, 'threshold': np.int64(10), 'left': np.float64(1.0), 'right': np.float64(5.0)}
[1. 1. 5. 5.]
[1 1] [5 5]
0.0


In [84]:
X = np.array([[1], [2], [10], [12]])
y = np.array([8, 8, 8, 8])
tree = build_tree(X, y, max_depth=3, min_samples_split=1)
print(tree)
preds = predict(X, tree)
print(preds)
threshold = 10
left = y[X[:,0] < threshold]
right = y[X[:,0] >= threshold]
print(left, right)
print(np.sum((left - left.mean())**2) + np.sum((right - right.mean())**2))

8.0
[8. 8. 8. 8.]
[8 8] [8 8]
0.0


In [85]:
X = np.array([[1], [2], [10], [12]])
y = np.array([1, 1, 5, 5])
tree = build_tree(X, y, min_samples_split=5)
print(tree)

3.0


In [86]:
tree = build_tree(X, y, max_depth=1)
print(tree)


{'feature': 0, 'threshold': np.int64(10), 'left': np.float64(1.0), 'right': np.float64(5.0)}
